# 02 — Attribution

Loads pre-extracted data, runs all registered attribution methods, and saves
results as a timestamped CSV in `results/`.

## How to add your own method
1. Implement it in the **Custom methods** cell below (a template is provided).
2. Register it in `ATTRIBUTION_METHODS` with a unique key.
3. Re-run the pipeline cells — your method will appear automatically in the output CSV.

The `ctx` dict available inside each lambda contains:
```
ctx['tas_f']    (T,)             factual temperature time series
ctx['gmt_f']    (T,)             smoothed factual GMT anomaly
ctx['gmt_c']    (T,)             smoothed counterfactual GMT
ctx['slp_f']    (T, n_lat, n_lon) global SLP field — slice as needed
ctx['slp_lat']  (n_lat,)         latitude axis
ctx['slp_lon']  (n_lon,)         longitude axis
ctx['ev_lat']   float            event barycentre latitude
ctx['ev_lon']   float            event barycentre longitude
ctx['val']      float            event threshold
ctx['range']    (t0, t1)         PN computation window
ctx['t_obs']    int              event time index
ctx['mth']      str              PN estimator set at runtime
```

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip install regionmask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 1.5 MB/s eta 0:00:00


In [3]:
import sys, os, pickle
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed

SRC_PATH = '/content/drive/MyDrive/ellis_ws_ch11-ml-attribution-extremes/ch11-ml-attribution-extremes/src'
sys.path.insert(0, SRC_PATH)

from attribution import (
    compute_pn,
    run_thermo_ml,
    run_dyn_adj_local_window,   # Ridge on local box, window only, no PCA
)
from analogues import (
    run_analogues,
    run_analogues_local,
    run_analogues_lasso,
    # run_analogues_causal,
    # run_analogues_ridge,
)
from data_utils import extract_local_slp


In [4]:
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.decomposition import PCA

def run_dyn_adj_local_window_slearner(gmt_f, gmt_cf, tas_f, slp_f, slp_lat, slp_lon, ev_lat, ev_lon,
                              obs_val, t_range, mth,
                              half_width_deg=25.0, alphas=np.logspace(-5, 15, 50), pca=False):
    """
    Dynamical adjustment on a raw local SLP box using Ridge regression

    Ridge is fitted on the full time series (1850-2014) so the regression
    captures the long-term SLP-temperature relationship and produces smooth,
    stable coefficients. PN is then computed on the window [t0, t1].

    Parameters
    ----------
    tas_f          : array (T,)               Full factual temperature series.
    slp_f          : array (T, n_lat, n_lon)  Full global SLP — data['f_slp']
    slp_lat        : array (n_lat,)           data['slp_lat']
    slp_lon        : array (n_lon,)           data['slp_lon']
    ev_lat         : float
    ev_lon         : float
    obs_val        : float
    t_range        : tuple (t0, t1)           Window used only for PN computation.
    mth            : str
    half_width_deg : float   Half-width in degrees (default 25.0 -> 50x50 total).
    alphas         : array-like

    Returns
    -------
    float  PN value.
    """
    slp_local = extract_local_slp(slp_f, slp_lat, slp_lon, ev_lat, ev_lon, half_width_deg)
    if pca:
      pca_ = PCA(n_components=5)
      slp_local = pca_.fit_transform(slp_local)
    regression_input = np.concatenate([gmt_f.reshape(-1, 1), slp_local], axis=1)
    reg       = RidgeCV(alphas=alphas).fit(regression_input, tas_f)   # full series fit
    counterfactual_reg_input = np.concatenate([gmt_cf.reshape(-1, 1), slp_local], axis=1)
    tas_dyn   = reg.predict(counterfactual_reg_input)
    # print("tas_dyn", tas_dyn)
    # print("tas_f", tas_f)
    print(gmt_f, gmt_cf)
    t0, t1    = t_range
    return compute_pn(tas_f[t0:t1], tas_dyn[t0:t1], obs_val, method=mth)

In [5]:
# ================================================================
#  CONFIG  —  edit here
# ================================================================
DATA_PATH   = '/content/drive/MyDrive/ellis_ws_ch11-ml-attribution-extremes/data/extracted_tasmax_nmem10_start1980_p99.9.pkl'
RESULTS_DIR = '/content/drive/MyDrive/ellis_ws_ch11-ml-attribution-extremes/results'

# PN estimators to run for every method
# Options: 'empirical', 'gaussian', 'gev'  (can select multiple)
STAT_METHODS = ['gaussian']

# Time window around each event
WINDOW_BEFORE = 72   # months
WINDOW_AFTER  = 12   # months

N_JOBS = os.cpu_count()
# ================================================================

## Built-in methods
These call functions from `src/attribution.py` and `src/analogues.py`.
Do not modify this cell — add your own methods in the **Custom methods** cell below.

In [12]:
ATTRIBUTION_METHODS = {

    # # ── Thermodynamic (GMT-based, no SLP) ──────────────────────────────────
    # 'pn_thermo_ML': lambda ctx:
    #     run_thermo_ml(
    #         ctx['tas_f'], ctx['gmt_f'], ctx['gmt_c'],
    #         ctx['val'], ctx['range'], ctx['mth']),

    # # ── Dynamical adjustment — local box, window only, no PCA ──────────────
    # # Ridge is fitted only on the time window [t0, t1] around the event,
    # # using the raw flattened local SLP box (half_width_deg on each side).
    # 'pn_dyn_adj_local_25': lambda ctx:
    #     run_dyn_adj_local_window(
    #         ctx['tas_f'], ctx['slp_f'],
    #         ctx['slp_lat'], ctx['slp_lon'],
    #         ctx['ev_lat'], ctx['ev_lon'],
    #         ctx['val'], ctx['range'], ctx['mth'],
    #         half_width_deg=12.5),

    # 'pn_dyn_adj_local_50': lambda ctx:
    #     run_dyn_adj_local_window(
    #         ctx['tas_f'], ctx['slp_f'],
    #         ctx['slp_lat'], ctx['slp_lon'],
    #         ctx['ev_lat'], ctx['ev_lon'],
    #         ctx['val'], ctx['range'], ctx['mth'],
    #         half_width_deg=25.0),

    # # ── Analogues — local SLP box (full record pool) ──────────────────────
    # 'pn_analogues_local': lambda ctx:
    #     run_analogues_local(
    #         ctx['tas_f'], ctx['gmt_f'], ctx['gmt_c'],
    #         ctx['slp_f'], ctx['slp_lat'], ctx['slp_lon'],
    #         ctx['ev_lat'], ctx['ev_lon'],
    #         ctx['val'], ctx['t_obs'], ctx['range'], ctx['mth'],
    #         half_width_deg=25.0, n_analogues=100),

    # # ── Analogues — Lasso-selected, global SLP flattened ───────────────────
    # 'pn_analogues_lasso': lambda ctx:
    #     run_analogues_lasso(
    #         ctx['tas_f'], ctx['gmt_f'], ctx['gmt_c'],
    #         ctx['slp_f'].reshape(ctx['slp_f'].shape[0], -1),
    #         ctx['val'], ctx['t_obs'], ctx['range'], ctx['mth'],
    #         n_years=50, n_analogues=50),

    # ── Add your method here ────────────────────────────────────────────────
    # 'pn_dyn_adj_local_50_slearner': lambda ctx:
    #     run_dyn_adj_local_window_slearner(
    #         ctx['gmt_f'], ctx['gmt_c'],
    #         ctx['tas_f'], ctx['slp_f'],
    #         ctx['slp_lat'], ctx['slp_lon'],
    #         ctx['ev_lat'], ctx['ev_lon'],
    #         ctx['val'], ctx['range'], ctx['mth'],
    #         half_width_deg=25.0,
    #         pca=False),
    'pn_dyn_adj_local_50_slearner_pca': lambda ctx:
        run_dyn_adj_local_window_slearner(
            ctx['gmt_f'], ctx['gmt_c'],
            ctx['tas_f'], ctx['slp_f'],
            ctx['slp_lat'], ctx['slp_lon'],
            ctx['ev_lat'], ctx['ev_lon'],
            ctx['val'], ctx['range'], ctx['mth'],
            half_width_deg=25.0,
            pca=True),

}


In [7]:
# ── Write your method below, then add it to ATTRIBUTION_METHODS ──────────

# def run_my_method(tas_f, slp_f, slp_lat, slp_lon,
#                   ev_lat, ev_lon, obs_val, t_range, mth):
#     slp_local = extract_local_slp(
#         slp_f, slp_lat, slp_lon, ev_lat, ev_lon, half_width_deg=20.0)
#     tas_cf = ...  # your counterfactual construction
#     t0, t1 = t_range
#     return compute_pn(tas_f[t0:t1], tas_cf[t0:t1], obs_val, method=mth)


# ── Register your method ─────────────────────────────────────────────────
# ATTRIBUTION_METHODS['pn_my_method'] = lambda ctx: run_my_method(
#     ctx['tas_f'], ctx['slp_f'], ctx['slp_lat'], ctx['slp_lon'],
#     ctx['ev_lat'], ctx['ev_lon'], ctx['val'], ctx['range'], ctx['mth'])

print('Custom methods registered:', [k for k in ATTRIBUTION_METHODS if k.startswith('pn_') and 'my' in k] or 'none yet')

Custom methods registered: none yet


In [13]:
def build_ctx(res, e_idx, is_factual):
    """Assemble the context dict passed to every attribution method."""
    tas_run = res['f_tas'] if is_factual else res['c_tas']
    slp_run = res['f_slp'] if is_factual else res['c_slp']
    idx_run = res['idx_f'] if is_factual else res['idx_c']
    t_obs   = idx_run[e_idx]
    t0      = max(0, t_obs - WINDOW_BEFORE)
    t1      = min(tas_run.shape[0], t_obs + WINDOW_AFTER)
    return {
        'tas_f':   tas_run[:, e_idx],
        'gmt_f':   res['gmt4_f'],
        'gmt_c':   np.full_like(res['gmt4_f'], res['gmt4_f'][0]),
        'slp_f':   slp_run,
        'slp_lat': res['slp_lat'],
        'slp_lon': res['slp_lon'],
        'ev_lat':  res['location'][e_idx, 0],
        'ev_lon':  res['location'][e_idx, 1],
        'val':     res['f_tas_vals'][e_idx] if is_factual else res['c_tas_vals'][e_idx],
        'range':   (t0, t1),
        't_obs':   t_obs,
    }


def process_event(m_idx, is_factual, data_list):
    """Run all methods on every event of one member-scenario."""
    res      = data_list[m_idx]
    scenario = 'factual' if is_factual else 'counterfactual'
    n_events = res['f_tas'].shape[1]
    rows     = []

    for e_idx in range(n_events):
        ctx = build_ctx(res, e_idx, is_factual)

        # Ensemble ground truth
        pool_f = np.concatenate([
            data_list[mi]['f_tas'][ctx['range'][0]:ctx['range'][1], e_idx]
            for mi in range(len(data_list))
            if e_idx < data_list[mi]['f_tas'].shape[1]
        ])
        pool_c = np.concatenate([
            data_list[mi]['c_tas'][ctx['range'][0]:ctx['range'][1], e_idx]
            for mi in range(len(data_list))
            if e_idx < data_list[mi]['c_tas'].shape[1]
        ])

        row = {
            'member':   res['member'],
            'event_id': e_idx,
            'scenario': scenario,
            'time':     res['times'][ctx['t_obs']],
            'lat':      ctx['ev_lat'],
            'lon':      ctx['ev_lon'],
        }

        for mth in STAT_METHODS:
            row[f'pn_ensemble_{mth}'] = compute_pn(pool_f, pool_c, ctx['val'], method=mth)
            for col_name, func in ATTRIBUTION_METHODS.items():
                ctx['mth'] = mth
                try:
                    row[f'{col_name}_{mth}'] = func(ctx)
                except Exception as exc:
                    print(f'[{res["member"]} e{e_idx} {mth} {col_name}] {exc}')
                    row[f'{col_name}_{mth}'] = np.nan

        rows.append(row)
    return rows

In [ ]:
with open(DATA_PATH, 'rb') as fh:
    data = pickle.load(fh)
print(f'Loaded {len(data)} members, {data[0]["f_tas"].shape[1]} events each.')

tasks   = [(i, fact) for i in range(len(data)) for fact in (True, False)]
# results = Parallel(n_jobs=N_JOBS)(
#     delayed(process_event)(m, f, data) for m, f in tqdm(tasks)
# )
process_event(0, True, data)

df = pd.DataFrame([row for sublist in results for row in sublist])
df['time'] = pd.to_datetime(df['time'])
print(f'Results shape: {df.shape}')
df.head()

Loaded 10 members, 147 events each.
tas_dyn [-0.09207745 -0.19570466 -0.33759718 ... -0.33775244 -0.62480764
 -0.34383953]
tas_f [ 2.0437489   1.733319   -1.081      ...  0.11110014  0.08290683
  0.51222765]
[-0.53707324 -0.53411446 -0.54181576 ...  1.08334212  1.07907857
  1.06995533] [-0.53707324 -0.53707324 -0.53707324 ... -0.53707324 -0.53707324
 -0.53707324]
tas_dyn [ 2.19090257  0.19690877 -0.83532742 ...  0.46034431 -2.17985125
 -0.07598122]
tas_f [ 3.0855608   0.94379866 -1.8330777  ...  1.0601149  -1.0310637
  0.5925815 ]
[-0.53707324 -0.53411446 -0.54181576 ...  1.08334212  1.07907857
  1.06995533] [-0.53707324 -0.53707324 -0.53707324 ... -0.53707324 -0.53707324
 -0.53707324]
tas_dyn [-0.69623957 -0.81830088 -0.65038593 ... -0.73228994 -0.71374331
 -0.51310672]
tas_f [-0.9178304 -1.0096294 -0.9975484 ...  1.4674561  0.592806   0.7316854]
[-0.53707324 -0.53411446 -0.54181576 ...  1.08334212  1.07907857
  1.06995533] [-0.53707324 -0.53707324 -0.53707324 ... -0.53707324 -0.53707

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
timestamp  = datetime.now().strftime('%Y%m%d_%H%M')
save_path  = os.path.join(RESULTS_DIR, f'attribution_{timestamp}.csv')
df.to_csv(save_path, index=False)
print(f'Saved -> {save_path}')